# Notebook 15 — Regression: Linear and Logistic

**Module:** 03 — Statistics and Probability  
**Notebook:** 15 of 18  
**Prerequisites:** NB04, NB05, NB09  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Regression is the backbone of quantitative biology: eQTL mapping (how much does a SNP
explain expression variance?), dose-response modeling, clinical outcome prediction,
DE analysis via generalized linear models (DESeq2, edgeR). Understanding regression
from first principles — not just calling `sklearn` — is a Track A interview requirement.

---
## Step 2 — Intuition

**Linear regression:** find the line $y = \beta_0 + \beta_1 x$ that minimizes the
sum of squared residuals. The Normal equations give a closed-form solution.

**Logistic regression:** predict a probability (0–1) from continuous predictors.
Squeeze the linear combination through a sigmoid function.
Trained by maximum likelihood, not least squares.

---
## Step 3 — Biological Background

**eQTL mapping:** is genotype at SNP $x$ a predictor of gene expression $y$?
Simple linear regression: $y_i = \beta_0 + \beta_1 x_i + \varepsilon_i$.
Testing $H_0: \beta_1 = 0$ is a t-test on the slope.

**DESeq2 under the hood:** fits a generalized linear model (negative binomial family)
for each gene. The DE test is a Wald test on the log2FC coefficient.

**Logistic regression in GWAS:** case-control GWAS fits logistic regression:
$\log(\text{odds}) = \beta_0 + \beta_1 \cdot \text{genotype}$.
$e^{\beta_1}$ is the odds ratio for the SNP.

---
## Step 4 — Mathematical Explanation

**OLS Normal equations:**
$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

**Slope for simple regression:**
$$\hat{\beta}_1 = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2} = \frac{\text{Cov}(x,y)}{\text{Var}(x)}$$

**Standard error of $\hat{\beta}_1$:**
$$\text{SE}(\hat{\beta}_1) = \frac{\hat{\sigma}}{\sqrt{\sum_i (x_i - \bar{x})^2}}, \quad \hat{\sigma}^2 = \frac{\sum_i \hat{\varepsilon}_i^2}{n-2}$$

**Logistic regression:**
$$P(y=1|x) = \sigma(\mathbf{x}^\top \boldsymbol{\beta}) = \frac{1}{1 + e^{-\mathbf{x}^\top \boldsymbol{\beta}}}$$

Log-likelihood:
$$\ell(\boldsymbol{\beta}) = \sum_i \left[ y_i \log \hat{p}_i + (1-y_i) \log(1-\hat{p}_i) \right]$$

Gradient (for gradient ascent):
$$\nabla_\beta \ell = \mathbf{X}^\top (\mathbf{y} - \hat{\mathbf{p}})$$

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — OLS from scratch: Normal equations
def ols_fit(X: np.ndarray, y: np.ndarray) -> dict:
    """
    Ordinary least squares via Normal equations.

    Parameters
    ----------
    X : design matrix (n_samples, n_features); should include intercept column
    y : response vector (n_samples,)

    Returns
    -------
    dict: coefficients, residuals, R^2, SE, t-stats, p-values
    """
    X, y = np.asarray(X), np.asarray(y)
    n, p = X.shape
    # Normal equations
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    y_hat = X @ beta
    residuals = y - y_hat
    # Variance estimate: RSS / (n - p) where p includes intercept
    sigma2 = (residuals**2).sum() / (n - p)
    # Covariance matrix of beta
    cov_beta = sigma2 * np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(cov_beta))
    t_stats = beta / se
    p_values = 2 * stats.t.sf(np.abs(t_stats), df=n - p)
    # R^2
    ss_tot = ((y - y.mean())**2).sum()
    ss_res = (residuals**2).sum()
    r2 = 1 - ss_res / ss_tot
    return dict(beta=beta, residuals=residuals, r2=r2, se=se,
                t_stats=t_stats, p_values=p_values, y_hat=y_hat, sigma2=sigma2)

# eQTL example: genotype (0/1/2 alleles) predicts expression
rng = np.random.default_rng(42)
n = 100
genotype = rng.choice([0, 1, 2], size=n, p=[0.25, 0.50, 0.25])
true_beta1 = 0.8  # allelic effect (log2 expression units)
expression = 5.0 + true_beta1 * genotype + rng.normal(0, 1, n)

X = np.column_stack([np.ones(n), genotype])  # design matrix with intercept
res = ols_fit(X, expression)

print("eQTL regression: expression ~ 1 + genotype")
print(f"  β₀ (intercept) = {res['beta'][0]:.3f}  SE={res['se'][0]:.3f}  p={res['p_values'][0]:.2e}")
print(f"  β₁ (genotype)  = {res['beta'][1]:.3f}  SE={res['se'][1]:.3f}  p={res['p_values'][1]:.2e}")
print(f"  R² = {res['r2']:.4f}")
print(f"  True β₁ = {true_beta1}")

In [ ]:
# Cell 6.2 — Logistic regression via gradient ascent
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-z))

def logistic_fit(X: np.ndarray, y: np.ndarray, lr: float = 0.1,
                  n_iter: int = 1000) -> dict:
    """
    Logistic regression via gradient ascent on log-likelihood.

    Parameters
    ----------
    X : design matrix including intercept column
    y : binary response (0 or 1)
    """
    X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
    n, p = X.shape
    beta = np.zeros(p)
    log_likelihoods = []
    for _ in range(n_iter):
        p_hat = sigmoid(X @ beta)
        gradient = X.T @ (y - p_hat)          # gradient of log-likelihood
        beta = beta + lr * gradient / n        # normalized step
        ll = np.sum(y * np.log(p_hat + 1e-15) + (1-y) * np.log(1-p_hat + 1e-15))
        log_likelihoods.append(ll)
    return dict(beta=beta, log_likelihoods=log_likelihoods)

# GWAS case-control: does SNP genotype predict disease?
n_case, n_ctrl = 200, 200
# Genotype frequency: allele frequency 0.3 (HWE)
geno_case = rng.binomial(2, 0.4, n_case)   # enriched in cases
geno_ctrl = rng.binomial(2, 0.3, n_ctrl)
geno_all = np.concatenate([geno_case, geno_ctrl])
y_all    = np.concatenate([np.ones(n_case), np.zeros(n_ctrl)])

X_all = np.column_stack([np.ones(len(geno_all)), geno_all])
lr_res = logistic_fit(X_all, y_all, lr=0.5, n_iter=500)

print("GWAS logistic regression: disease ~ 1 + genotype")
print(f"  β₀ = {lr_res['beta'][0]:.4f}")
print(f"  β₁ = {lr_res['beta'][1]:.4f}  (odds ratio = {np.exp(lr_res['beta'][1]):.3f})")

# Compare to sklearn for validation
from sklearn.linear_model import LogisticRegression
sk_lr = LogisticRegression(C=1e6, fit_intercept=False, max_iter=1000)
sk_lr.fit(X_all, y_all)
print(f"\nsklearn coefficients: {sk_lr.coef_[0]}")
print(f"Our coefficients:     {lr_res['beta']}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — OLS: scatter + fit + residuals; Logistic: sigmoid + log-likelihood convergence
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: OLS eQTL fit
ax = axes[0]
ax.scatter(genotype, expression, s=15, alpha=0.5, color="steelblue")
x_line = np.array([0, 1, 2])
y_line = res['beta'][0] + res['beta'][1] * x_line
ax.plot(x_line, y_line, 'tomato', lw=2, label=f"β₁={res['beta'][1]:.3f}")
ax.set_xlabel("Genotype (0/1/2 alleles)"); ax.set_ylabel("log₂ Expression")
ax.set_title(f"eQTL: OLS regression\nR²={res['r2']:.3f}")
ax.legend(fontsize=8)

# Panel 2: Logistic sigmoid
ax = axes[1]
x_sig = np.linspace(-3, 3, 200)
# Use fitted coefficients to draw decision boundary
b0, b1 = lr_res['beta']
prob_geno = [sigmoid(b0 + b1 * g) for g in [0, 1, 2]]
jitter = rng.normal(0, 0.02, len(geno_all))
ax.scatter(geno_all, y_all + jitter, s=8, alpha=0.2, color="steelblue")
geno_range = np.linspace(-0.2, 2.2, 200)
ax.plot(geno_range, sigmoid(b0 + b1 * geno_range), 'tomato', lw=2,
        label=f"OR={np.exp(b1):.2f}")
ax.set_xlabel("Genotype"); ax.set_ylabel("P(case)")
ax.set_title("GWAS logistic regression")
ax.legend(fontsize=8)

# Panel 3: log-likelihood convergence
ax = axes[2]
ax.plot(lr_res['log_likelihoods'], color="steelblue", lw=1.5)
ax.set_xlabel("Iteration"); ax.set_ylabel("Log-likelihood")
ax.set_title("Logistic regression convergence")

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `ols_fit()` from scratch using the Normal equations. Verify $\hat{\beta}$
   against `np.polyfit` and check that residuals sum to zero.
2. Derive the gradient of the logistic log-likelihood:
   $\nabla_\beta \ell = \mathbf{X}^\top (\mathbf{y} - \hat{\mathbf{p}})$.
   Start from the log-likelihood expression and differentiate with respect to $\beta$.
3. In the eQTL example: increase the noise level ($\sigma$). At what noise level does
   the genotype effect become non-significant (p > 0.05)?
4. What is $R^2$? What does it measure? Can $R^2$ be negative for OLS? When?
   (Hint: is it guaranteed by the Normal equations to be non-negative?)

---
## Quiz — Active Recall

1. Write the Normal equations for OLS.
2. What is the gradient of the logistic log-likelihood? Derive it in one line.
3. In GWAS logistic regression, what does the exponentiated coefficient $e^{\beta_1}$ represent?
4. What is $R^2$? What values can it take? When is it 0? When is it 1?
5. Why does DESeq2 use a negative binomial GLM rather than linear regression?

---
## Reflection

**Date completed:** ____________________

> *[Could you derive the OLS Normal equations from first principles? Could you implement logistic regression gradient ascent without notes?]*

---
*Next: `16_statistics_on_expression_data.ipynb`*